In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

path = "../data/raw/Mobile_Parts_Wholesale_Dataset.xlsx"
sales = pd.read_excel(path, sheet_name="Sales_Transactions")
sales['order_date'] = pd.to_datetime(sales['order_date'])

snapshot_date = sales['order_date'].max() + pd.Timedelta(days=1)

rfm = sales.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_amount_inr', 'sum')
).reset_index()

print(rfm.shape)
rfm.head()

(179, 4)


,customer_id,recency,frequency,monetary
0,C0001,1,9,19979.0
1,C0002,6,32,141835.5
2,C0003,19,10,36461.0
3,C0004,43,59,257908.5
4,C0005,1,18,37232.5


In [2]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['segment'] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby('segment')[['recency','frequency','monetary']].mean()
print(cluster_summary)

           recency  frequency       monetary
segment                                     
0        13.379310  60.931034  261585.551724
1         8.368421  39.912281  160288.666667
2        13.628571  17.071429   65361.464286
3        46.304348  33.869565  122304.978261


D:\dataScience\envs\dss_project\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [3]:
segment_labels = {0: "High Value", 1: "Steady/Regular", 2: "New/Occasional", 3: "At Risk"}
rfm['segment_label'] = rfm['segment'].map(segment_labels)

rfm.to_csv("../reports/customer_segments.csv", index=False)
print("Saved customer_segments.csv")
rfm['segment_label'].value_counts()

Saved customer_segments.csv


segment_label
New/Occasional    70
Steady/Regular    57
High Value        29
At Risk           23
Name: count, dtype: int64